In [0]:
%sql
CREATE OR REPLACE TABLE silver_catalog.silver.silverNetflixTitles AS
SELECT
  id,
  nullif(trim(title), '')              AS title,
  nullif(trim(type), '')               AS type,
  nullif(trim(description), '')        AS description,

  releaseYear,

  nullif(trim(ageCertification), '')   AS ageCertification,

  runTime,
  CAST(regexp_extract(runTime, '(\\d+)', 1) AS INT) AS runTimeMinutes,

--  list-like strings → arrays
  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(genres, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS genresArr,

  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(productionCountries, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS productionCountriesArr,

  seasons,

  nullif(trim(imdbId), '')             AS imdbId,
  imdbScore,
  CAST(imdbVotes AS INT)              AS imdbVotes,        -- make votes integer
  tmdbPopularity,
  tmdbScore,

  rescuedData,

  -- audit / lineage
  sourceFilePath,
  ingestTs

FROM bronze_landing.media_analytics.bronzeNetflixTitles;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_catalog.silver.silverImdbAdvancedMoviesDetails AS
SELECT
  link,

  -- list-like strings → arrays
  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(writers, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS writersArr,

  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(directors, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS directorsArr,

  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(stars, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS starsArr,

  -- keep raw money strings (safe)
  budget,
  openingWeekendGross,
  grossWorldwide,
  grossUsCanada,

  -- keeping just as digits
  CAST(regexp_replace(budget, '[^0-9]', '') AS BIGINT) AS budgetAmount,
  CAST(regexp_replace(openingWeekendGross, '[^0-9]', '') AS BIGINT) AS openingWeekendGrossAmount,
  CAST(regexp_replace(grossWorldwide, '[^0-9]', '') AS BIGINT) AS grossWorldwideAmount,
  CAST(regexp_replace(grossUsCanada, '[^0-9]', '') AS BIGINT) AS grossUsCanadaAmount,

  releaseDate,

  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(countriesOrigin, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS countriesOriginArr,

  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(filmingLocations, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS filmingLocationsArr,

  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(productionCompany, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS productionCompanyArr,

  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(awardsContent, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS awardsContentArr,

  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(genres, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS genresArr,

  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(languages, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS languagesArr,

  -- lineage / audit
  fileYear,
  sourceFilePath,
  ingestTs

FROM bronze_landing.media_analytics.bronzeImdbAdvancedMoviesDetails;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_catalog.silver.silverImdbMovies AS
SELECT
  nullif(trim(title), '') AS title,

  year AS releaseYear,

  duration,
  try_cast(regexp_extract(duration, '(\\d+)', 1) AS INT) AS runTimeMinutes,

  nullif(trim(mpa), '') AS ageCertification,

  -- rating & votes are strings in bronze → type them in silver
  CAST(regexp_extract(rating, '(\\d+(?:\\.\\d+)?)', 1) AS DOUBLE) AS imdbScore,

  CAST(regexp_replace(votes, '[^0-9]', '') AS INT) AS imdbVotes,

  metaScore,
  nullif(trim(description), '') AS description,
  movieLink,

  -- lineage / audit
  fileYear,
  sourceFilePath,
  ingestTs

FROM bronze_landing.media_analytics.bronzeImdbMovies;

In [0]:
%sql
CREATE OR REPLACE TABLE silver_catalog.silver.silverImdbMergedMoviesData AS
SELECT
  nullif(trim(title), '') AS title,
  year AS releaseYear,

  duration,
  try_cast(regexp_extract(duration, '(\\d+)', 1) AS INT) AS runTimeMinutes,

  nullif(trim(mpa), '') AS ageCertification,

  rating AS imdbScore,

  CAST(regexp_replace(votes, '[^0-9]', '') AS INT) AS imdbVotes,

  metaScore,
  nullif(trim(description), '') AS description,
  movieLink,

  -- list-like strings -> arrays
  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(writers, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS writersArr,

  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(directors, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS directorsArr,

  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(stars, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS starsArr,

-- strings
  budget,
  openingWeekendGross,
  grossWorldwide,
  grossUsCanada,

  -- keeping just as digits
  CAST(regexp_replace(budget, '[^0-9]', '') AS BIGINT) AS budgetAmount,
  CAST(regexp_replace(openingWeekendGross, '[^0-9]', '') AS BIGINT) AS openingWeekendGrossAmount,
  CAST(regexp_replace(grossWorldwide, '[^0-9]', '') AS BIGINT) AS grossWorldwideAmount,
  CAST(regexp_replace(grossUsCanada, '[^0-9]', '') AS BIGINT) AS grossUsCanadaAmount,

  releaseDate,

  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(countriesOrigin, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS countriesOriginArr,

  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(filmingLocations, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS filmingLocationsArr,

  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(productionCompany, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS productionCompanyArr,

  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(awardsContent, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS awardsContentArr,

  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(genres, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS genresArr,

  FILTER(
    TRANSFORM(
      SPLIT(
        REGEXP_REPLACE(REGEXP_REPLACE(languages, '^\\[|\\]$', ''), '''', ''),
        ',\\s*'
      ),
      x -> TRIM(x)
    ),
    x -> x <> ''
  ) AS languagesArr,

  -- lineage / audit
  fileYear,
  sourceFilePath,
  ingestTs

FROM bronze_landing.media_analytics.bronzeImdbMergedMoviesData;